# Hyperparameter Tuning for Regression Models

## Definition

**Hyperparameter tuning** is the process of optimizing the non-trainable configuration settings of a machine learning algorithm to maximize its predictive performance.

Unlike model parameters—such as weights in linear regression or split points in decision trees—which are learned automatically from data during training, **hyperparameters** must be specified before the training process begins.

In regression models, hyperparameter tuning directly controls model complexity, balancing the trade-off between **underfitting** (high bias) and **overfitting** (high variance).

---

## Why It Matters

Default hyperparameter configurations provided by machine learning libraries are general-purpose baselines and rarely yield optimal performance for a specific dataset. Improperly tuned hyperparameters lead to two primary failure modes:

* **Overfitting**: The model becomes overly complex, memorizing training data noise and failing to generalize to unseen test data.


* **Underfitting**: The model is overly constrained, failing to capture underlying trends and relationships in the data.



Proper tuning ensures that:

1. **Regularization penalties** in linear models appropriately constrain coefficient size without destroying predictive signal.


2. **Tree ensembles** (such as Random Forest and Gradient Boosting) achieve optimal depth, sample splitting, and ensemble size to maximize predictive accuracy.



---

## Tuning Linear Models ($L_1$ and $L_2$ Regularization)

Regularized linear models control complexity by adding a mathematical penalty to the Ordinary Least Squares (OLS) loss function. Tuning these models primarily involves optimizing the strength of this penalty.

### 1. Ridge Regression ($L_2$ Regularization)

Ridge regression adds a penalty proportional to the square of the magnitude of the feature coefficients:

$$\text{Loss}_{\text{Ridge}} = \text{MSE} + \alpha \sum_{j=1}^{p} w_j^2$$

Where:

* $\text{MSE}$ is the Mean Squared Error.


* $w_j$ represents individual feature weights/coefficients.


* $\alpha$ (alpha, or $\lambda$ in some literature) is the **hyperparameter** controlling penalty strength.



**Key Hyperparameters to Tune**:

* `alpha`: Controls regularization strength.
* **$\alpha = 0$**: Equivalent to standard OLS Linear Regression.


* **Small $\alpha$**: Low penalty; higher risk of overfitting.


* **Large $\alpha$**: Heavy penalty; shrinks coefficients toward zero, reducing variance but risking underfitting.





### 2. Lasso Regression ($L_1$ Regularization)

Lasso regression adds a penalty proportional to the absolute value of the feature coefficients:

$$\text{Loss}_{\text{Lasso}} = \text{MSE} + \alpha \sum_{j=1}^{p} \vert{}w_j\vert{}$$

**Key Hyperparameters to Tune**:

* `alpha`: Controls regularization strength.
* **Impact**: Because of the absolute value geometry, increasing `alpha` forces coefficients of less important features to **exactly zero**, performing automated feature selection.




* `max_iter`: Maximum number of iterations for the optimization solver to converge.
* **Impact**: Must be increased when `alpha` is very small, as the optimization algorithm requires more iterations to converge.





### 3. ElasticNet

ElasticNet combines both $L_1$ and $L_2$ penalties into a single loss function:

$$\text{Loss}_{\text{ElasticNet}} = \text{MSE} + \alpha \cdot l1\_ratio \sum_{j=1}^{p} \vert{}w_j\vert{} + \frac{\alpha (1 - l1\_ratio)}{2} \sum_{j=1}^{p} w_j^2$$

**Key Hyperparameters to Tune**:

* `alpha`: Controls overall penalty magnitude.


* `l1_ratio`: Controls the mix of $L_1$ and $L_2$ regularization.
* **`l1_ratio = 1.0`**: Pure Lasso ($L_1$).


* **`l1_ratio = 0.0`**: Pure Ridge ($L_2$).


* $0.0 < \mathtt{l1\_ratio} < 1.0$: A hybrid combination, effective when features are correlated.
---

## Tuning Tree-Based Ensembles (Bagging and Boosting)

Tree-based models do not use coefficient penalties. Instead, their complexity is tuned by constraining individual tree architecture or controlling how trees are combined.

### 1. Bagging Models (Random Forest Regressor)

Random Forests build multiple decision trees independently in parallel using bootstrap aggregation (bagging) and average their predictions.

**Key Hyperparameters to Tune**:

* `n_estimators`: The number of decision trees in the forest.


* **Tuning Strategy**: Higher is generally better for stability, but increases computation time. It exhibits diminishing returns and does **not** cause overfitting.


* `max_depth`: The maximum allowed depth of each individual decision tree.


* **Tuning Strategy**: Deep trees capture complex patterns but overfit easily. Restricting depth prevents individual trees from memorizing training data noise.




* `min_samples_split`: The minimum number of samples required to split an internal node.


* **Tuning Strategy**: Higher values prevent the model from creating splits based on tiny, noisy data samples.




* `min_samples_leaf`: The minimum number of samples required to be present at a leaf node.


* **Tuning Strategy**: Higher values smooth predictions, particularly in noisy regression regions.




* `max_features`: The fraction or number of random features evaluated for each split.


* **Tuning Strategy**: Reduces correlation between individual trees, lowering total ensemble variance.





### 2. Boosting Models (Gradient Boosted Trees / XGBoost / LightGBM)

Boosting models build decision trees sequentially, where each new tree is trained to fit the residual errors of the combined preceding trees.

**Key Hyperparameters to Tune**:

* `learning_rate` (or `eta`): Scales the contribution of each new tree added to the ensemble.


* **Tuning Strategy**: Smaller values ($\le 0.1$) require more trees (`n_estimators`) but lead to better generalization. High learning rates cause the model to overcorrect, leading to instability.


* `n_estimators`: The number of sequential trees to build.


* **Tuning Strategy**: Unlike Random Forest, setting `n_estimators` too high in boosting **will cause overfitting** if the learning rate is not properly balanced.




* `max_depth`: Depth of individual sequential trees.


* **Tuning Strategy**: In boosting, individual trees should remain shallow (often depth 3–6) to act as "weak learners."


* `subsample`: The fraction of training samples randomly drawn to build each tree.


* **Tuning Strategy**: Setting values below $1.0$ (e.g., $0.8$) adds stochasticity, preventing trees from focusing excessively on specific outliers.





---

## Hyperparameter Search Strategies

To determine optimal hyperparameter combinations, data scientists apply structured search techniques evaluated via **Cross-Validation (CV)**.

```
 Full Hyperparameter Search Space
  │
  ├──► Grid Search ───► Evaluates EVERY fixed combination in a grid (Exhaustive)
  │
  ├──► Random Search ──► Evaluates a RANDOM sample of combinations (Efficient)
  │
  └──► Bayesian Search ─► Models past evaluation metrics to pick the NEXT best candidates

```

| Search Strategy | Description | Best Used When |
| --- | --- | --- |
| **Grid Search** (`GridSearchCV`) | Systematically evaluates every single candidate combination within a specified parameter grid.| The hyperparameter search space is small and evaluation is computationally cheap.|
| **Randomized Search** (`RandomizedSearchCV`) | Randomly samples a fixed number of parameter combinations from statistical distributions. | The search space is high-dimensional or training time per model is long. |
| **Bayesian Optimization** | Fits a probabilistic model to past evaluation results to intelligently select the next best parameters to test. | Complex gradient boosted models require fine-grained tuning across many continuous hyperparameters. |

---

## Common Mistakes

* **Tuning on the Test Set**: Performing hyperparameter searches directly on the test set leads to data leakage, producing overly optimistic performance estimates that fail on real-world data.


* **Searching Linearity on Unscaled Features**: $L_1$ and $L_2$ regularization penalize large coefficients. If input features are on different scales, the penalty will unfairly affect larger-scale features regardless of their true predictive power.


* **Tuning `n_estimators` Independently in Boosting**: In boosting algorithms, `learning_rate` and `n_estimators` are inversely linked. Tuning one without adjusting the other results in poor convergence or severe overfitting.


* **Using Unconstrained Tree Depth in Ensembles**: Allowing individual trees in a Random Forest or Gradient Boosting model to grow without limits significantly increases memory usage and leads to overfitting on noisy regression targets.



---

## Quick Recap

* **Hyperparameters** control algorithm structure and complexity; they are set *before* model training begins.


* **$L_1$ (Lasso) and $L_2$ (Ridge)** regularization manage linear model complexity via penalty strength `alpha`.


* **Random Forest (Bagging)** benefits from deeper trees and high `n_estimators`, focusing on variance reduction.


* **Gradient Boosting** relies on a tight balance between small `learning_rate`, shallow `max_depth`, and moderate `n_estimators`.


* **Cross-validation** combined with **Randomized or Grid Search** prevents data leakage while systematically discovering optimal hyperparameter setups.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Reading the excel file
data = pd.read_excel("data/House Price Prediction.xlsx")

In [3]:
data

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,20140521T000000,360000,3,2.50,1530,1131,3.0,0,0,...,8,1530,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,6600060120,20150223T000000,400000,4,2.50,2310,5813,2.0,0,0,...,8,2310,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,1523300141,20140623T000000,402101,2,0.75,1020,1350,2.0,0,0,...,7,1020,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,291310100,20150116T000000,400000,3,2.50,1600,2388,2.0,0,0,...,8,1600,0,2004,0,98027,47.5345,-122.069,1410,1287


### Data Cleaning

In [4]:
print(f"Rows: {data.shape[0]}\nColumns: {data.shape[1]}")

Rows: 21613
Columns: 21


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21613 non-null  int64  
 1   date           21613 non-null  str    
 2   price          21613 non-null  int64  
 3   bedrooms       21613 non-null  int64  
 4   bathrooms      21613 non-null  float64
 5   sqft_living    21613 non-null  int64  
 6   sqft_lot       21613 non-null  int64  
 7   floors         21613 non-null  float64
 8   waterfront     21613 non-null  int64  
 9   view           21613 non-null  int64  
 10  condition      21613 non-null  int64  
 11  grade          21613 non-null  int64  
 12  sqft_above     21613 non-null  int64  
 13  sqft_basement  21613 non-null  int64  
 14  yr_built       21613 non-null  int64  
 15  yr_renovated   21613 non-null  int64  
 16  zipcode        21613 non-null  int64  
 17  lat            21613 non-null  float64
 18  long           21

In [6]:
data.isna().sum()

id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

In [7]:
print(f"Total Duplicates in dataset: {data.duplicated().sum()}")

Total Duplicates in dataset: 0


In [8]:
data = data.drop(columns=["id", "date"])

### Feature and Target Selection

In [9]:
x = data.drop(columns="price")
y = data["price"]

### Splitting the dataset

In [10]:
from sklearn.model_selection import train_test_split
x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.5, random_state=42)

In [11]:
print(data.shape)
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(21613, 19)
(15129, 18)
(3242, 18)
(3242, 18)


### Data Preprocessing

In [12]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

### Model Training

In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

rfr_model = RandomForestRegressor()
lr_model = LinearRegression()
l1_model = Lasso(alpha=1.0) # Pure l1
l2_model = Ridge(alpha=1.0) # Pure l2

In [19]:
rfr_model.fit(x_train_scaled, y_train)
lr_model.fit(x_train_scaled, y_train)
l1_model.fit(x_train_scaled, y_train)
l2_model.fit(x_train_scaled, y_train)

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.785e+13, tolerance: 1.977e+11
  model = cd_fast.enet_coordinate_descent(


Ridge()

In [21]:
l1_model.coef_

array([-32037.28956448,  30873.43645156, 257426.20252056,   3908.51491065,
         2862.60293567,  45576.35935354,  41864.29614948,  15073.36078746,
       111379.35712967, -81195.56756122, -57277.17879042, -76178.28983438,
         8705.81882305, -29676.83168382,  82471.73100859, -27547.14171346,
        11074.59787979,  -9020.60505322])

### Model Tuning

In [22]:
rfr_pred = rfr_model.predict(x_val_scaled)
lr_pred= lr_model.predict(x_val_scaled)
l1_pred= l1_model.predict(x_val_scaled)
l2_pred= l2_model.predict(x_val_scaled)

In [23]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

print(f"MAE of Random Forest: {mean_absolute_error(y_val, rfr_pred)}")
print(f"MAPE of Random Forest: {mean_absolute_percentage_error(y_val, rfr_pred)*100}")

print(f"MAE of Linear Regression: {mean_absolute_error(y_val, lr_pred)}")
print(f"MAPE of Linear Regression: {mean_absolute_percentage_error(y_val, lr_pred)*100}")

print(f"MAE of Lasso Regression: {mean_absolute_error(y_val, l1_pred)}")
print(f"MAPE of Lasso Regression: {mean_absolute_percentage_error(y_val, l1_pred)*100}")

print(f"MAE of Ridge Regression: {mean_absolute_error(y_val, l2_pred)}")
print(f"MAPE of Ridge Regression: {mean_absolute_percentage_error(y_val, l2_pred)*100}")

MAE of Random Forest: 72280.48494459622
MAPE of Random Forest: 13.592678488698635
MAE of Linear Regression: 124839.56999347289
MAPE of Linear Regression: 25.337793260083473
MAE of Lasso Regression: 124839.58605337658
MAPE of Lasso Regression: 25.337733024515526
MAE of Ridge Regression: 124837.43626159815
MAPE of Ridge Regression: 25.337027858023042


### Model Evaluation

In [24]:
pred_rfr = rfr_model.predict(x_test_scaled)
pred_lr = lr_model.predict(x_test_scaled)

In [25]:
print(f"MAE of Random Forest: {mean_absolute_error(y_test, pred_rfr)}")
print(f"MAPE of Random Forest: {mean_absolute_percentage_error(y_test, pred_rfr)*100}")
print(f"MAE of Linear Regression: {mean_absolute_error(y_test, pred_lr)}")
print(f"MAPE of Linear Regression: {mean_absolute_percentage_error(y_test, pred_lr)*100}")

MAE of Random Forest: 75285.5041904101
MAPE of Random Forest: 13.440434920226064
MAE of Linear Regression: 130134.03512089474
MAPE of Linear Regression: 25.271175432793342
